# run_experiment

One experiment, end to end, from a settings file. Works on Colab and on a rented
GPU box; the only difference is which settings file you name.

Edit the first code cell and nothing else: the settings file, the branch, and the Hub repo. To run a second experiment, change
`SETTINGS_FILE` and re-run from **Run the experiment** downwards. The model stays
downloaded and the environment stays installed, so each extra run costs only its
own training time.

On a rented machine that expires, run the save cell after every experiment rather
than at the end. Everything on that box is deleted when it shuts down.

In [ ]:
USE_TEST_CONFIG = False         # True: the CPU smoke settings in src/configs/test.py
SETTINGS_FILE = "configs/minimum_threshold_one_persona/rule_tests/persona_a/one_persona_minimum_threshold_cut25.json"

# Resume an interrupted run instead of starting a new one (README "Resuming a run"). Give the
# run's id, its folder under results/, and a decision step it saved a checkpoint at and finished
# evaluating. It keeps the run's id and folders, skips the work already done, finishes an
# introspection branch at that step that never did, and trains on from the next step. Only
# compute, gates, training_steps and later introspection points may differ from the run's own
# settings; anything else is refused. None starts a new run. For the run that ran out of memory:
#   SETTINGS_FILE = "configs/paper_minimum_threshold/paper_minimum_threshold_8b_16gb_resume.json"
#   RESUME_RUN_ID = "20261031-061400_paper_minimum_threshold_Qwen3-8B"
#   RESUME_FROM_STEP = 1500
RESUME_RUN_ID = None
RESUME_FROM_STEP = None

# The branch of the repo to run. On a fresh machine it is cloned; an existing clone is moved
# onto it and brought up to date, so the code is always this branch's latest commit.
BRANCH = "bootstrap-ci"

# Where finished runs are saved so they survive the machine. Must be a Hub repo you
# can write to; the owner must match the account your token belongs to.
HUB_REPO = "adamchanna/bdp-plunkett-qwen3"

# Settings files: configs/README.md maps every folder. Each file is complete (nothing inherited),
# so open it to see, or edit, every value the run uses.
#   configs/default.json                         Plunkett on Qwen3-0.6B, 4-bit, fp16 (Colab T4)
#   configs/plunkett_linear/                     atkinson, linear-4b_20gb, linear-8b_20gb
#   configs/minimum_threshold/                   the minimum threshold rule at Qwen3-8B, many personas
#   configs/paper_minimum_threshold/             the paper's design: 8b, 8b_16gb (+ _units, _resume), 0.6b, 1500_steps
#   configs/minimum_threshold_one_persona/       one persona, Qwen3-8B: base/ controls/ inverse/ rule_tests/ cut_sweep/,
#                                                each split into persona_a/ and persona_b/. Every experiment has a
#                                                16 GB file (5070 Ti), a _24gb file (L4, A10, 4090), a _40gb file (A100 40GB) and an _80gb file (A100 80GB)
#   configs/minimum_threshold_one_persona_qwen3_14b/   the same on Qwen3-14B: _14b_24gb, _14b_80gb
#   configs/minimum_threshold_one_persona_qwen3_32b/   the same on Qwen3-32B: _32b_48gb, _32b_80gb
#
# A card's sizing is micro_batch_size and the compute eval batch sizes, in the file itself.
#
# Introspection, set in the settings file under model_hyperparameters (README "Settings"):
#   introspection_at_steps    the decision steps stage two branches from. A list, [1500, 2500, 3000],
#                             or a range, {"start": 1000, "end": 3000, "every": 500}, which is
#                             1000, 1500, ..., 3000. null: the last decision step only
#   introspection_steps       how long each branch trains (its end point), per fold
#   introspection_batch_size  examples per introspection step
#   introspection_training    false switches stage two off altogether
#   training_steps            the last decision step; no branch point may lie past it
#   introspection_targets     what each branch trains toward, one branch each: hidden (the persona's
#                             own rule), recovered (the rule read from the model's own choices),
#                             shuffled (another persona's: format without content), random
#   introspection_remeasure_decisions  ask the held-out personas' decisions again after each branch
#   lr_schedule               null (constant) or {"type": "linear"/"cosine", "warmup_steps": W,
#                             "final_fraction": f}
#
# Which questions are asked and trained on: report_schema.questions in the settings file, each
# {"ask": true/false, "train": true/false}; every question is a folder under questions/ (README "Questions").


## Set up the machine

In [ ]:
!nvidia-smi || echo "no nvidia-smi: CPU-only machine, use USE_TEST_CONFIG = True"

In [ ]:
# Clone BRANCH when the code is not already here, or move an existing clone onto it, then pin versions.
# run_me.py runs before the install: it re-pins requirements to whatever this machine
# already ships, keeping its CUDA-matched torch instead of letting pip swap it out.
import os, subprocess, sys

CLONE_URL = "https://github.com/Bilal-Trigui/Decision_Task_Database_Experiments.git"
if not os.path.exists("src"):
    if not os.path.exists("Decision_Task_Database_Experiments"):
        subprocess.run(["git", "clone", "-b", BRANCH, CLONE_URL], check=True)
    os.chdir("Decision_Task_Database_Experiments")
# An existing clone may sit on another branch or an older commit. This refuses rather than
# overwrite local edits: commit or stash them, or set BRANCH to the branch they are on.
subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(f"code: branch {BRANCH}, commit {commit}")
# Forget any copy of the project's code this session imported earlier, so the code just pulled is the code that
# runs. Without this, running this cell again after a push goes on with the old code until the session is restarted.
for module in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[module]
subprocess.run([sys.executable, "run_me.py"], check=True)

# Keep model downloads in one folder, easy to delete on a shared disk.
os.environ.setdefault("HF_HOME", os.path.abspath("hf_cache"))
# Let the CUDA allocator grow segments instead of fragmenting. Must be set before
# torch initialises CUDA, which is why it lives here and not in the GPU check.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("repo ready in", os.getcwd())

In [ ]:
# %pip installs into THIS kernel's environment. A pip subprocess can land packages
# somewhere the kernel cannot import from, which surfaces later as ModuleNotFoundError.
%pip install -q -r requirements.txt

In [ ]:
# Confirm the kernel can import what the pipeline needs.
#
# On JupyterHub, pip usually installs into the user site-packages, and a kernel
# that started before that folder existed does not have it on sys.path. That is
# the normal cause of ModuleNotFoundError straight after a successful install.
# Adding the folder here repairs it without a restart.
import importlib, importlib.util, os, site, sys

user_site = site.getusersitepackages()
if user_site and os.path.isdir(user_site) and user_site not in sys.path:
    sys.path.append(user_site)
    importlib.invalidate_caches()
    print("added user site-packages to sys.path:", user_site)

print("kernel python:", sys.executable)
needed = ("numpy", "pandas", "sklearn", "matplotlib", "torch", "transformers", "peft", "huggingface_hub")
missing = [m for m in needed if importlib.util.find_spec(m) is None]
if missing:
    print("MISSING:", missing)
    print("Restart the kernel (Kernel -> Restart Kernel), run the install cell, then this cell again.")
else:
    print("all imports available")


In [ ]:
# Confirm the GPU, and say which one-persona settings files are sized for it. bf16 needs Ampere or
# newer: the bf16 settings files ask for it and the loader refuses rather than falling back.
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print("gpu", torch.cuda.get_device_name(0))
    print("vram GB", round(vram, 1))
    print("bf16 supported", torch.cuda.is_bf16_supported())
    # A card reports a little under its name (an L4 about 22.5 GB, a 16 GB card about 15.9), so a card counts
    # as big enough from 90% of the size a file is sized for.
    cards = {"the file with no suffix (16 GB)": 16, "_24gb": 24, "_40gb": 40, "_80gb": 80}
    fit = [name for name, gb in cards.items() if gb * 0.9 <= vram]
    if fit:
        print("\none-persona Qwen3-8B settings files this card can run: " + ", ".join(fit))
        print(f"sized for this card: {fit[-1]}" + (". A smaller one runs too, with smaller evaluation batches, "
              "so each checkpoint is scored more slowly." if len(fit) > 1 else ""))
    else:
        print(f"\nthis card has {vram:.0f} GB, less than the 16 GB the smallest one-persona settings file is sized for")
else:
    print("no CUDA device: only USE_TEST_CONFIG = True will run here")

In [ ]:
# Credentials. Typed rather than written into a cell, so nothing lands in git or in
# a saved notebook. Needed only for the save cells; the public Qwen3 weights do not
# require a token.
#
# A token that the Hub rejects is discarded and you are asked again, so a typo or
# a revoked token never gets stuck in the environment.
import getpass, os
from huggingface_hub import HfApi
from huggingface_hub.utils import HfHubHTTPError

owner = HUB_REPO.split("/")[0]
who = None
for attempt in range(3):
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face WRITE token (blank to skip): ")
    if not token:
        break
    try:
        who = HfApi(token=token).whoami()
        os.environ["HF_TOKEN"] = token
        break
    except HfHubHTTPError as err:
        os.environ.pop("HF_TOKEN", None)
        if err.response is not None and err.response.status_code == 401:
            print("The Hub rejected that token (401). Check it is a WRITE token and not revoked. Try again.")
            continue
        raise

if who is None:
    print("no token: the run works, but saving refuses and nothing survives the machine")
else:
    allowed = [who["name"]] + [o["name"] for o in who.get("orgs", [])]
    print("token belongs to:", who["name"])
    if owner in allowed:
        print(f"HUB_REPO owner '{owner}' matches. Saving will work.")
    else:
        print(f"MISMATCH: HUB_REPO starts with '{owner}' but this token can only write to {allowed}.")
        print(f"Set HUB_REPO = \"{who['name']}/bdp-plunkett-qwen3\" in the first cell and re-run it.")


## Run the experiment

On a rented box, go in this order and save after each one. Starting small
exercises the CUDA path, the estimator, the report parser and the results writer in
a few minutes, which is the cheapest place to find a broken setting.

    default  ->  linear-4b_20gb  ->  linear-8b_20gb

For the minimum threshold experiments, the 0.6B run first, then 8B:

    paper_minimum_threshold_0.6b  ->  paper_minimum_threshold_8b

The cell prints the plan before training starts: the rule, the model, the decision steps,
where introspection branches and for how long. Check it before walking away.

In [ ]:
from src.config import load
from src.pipeline import run
from src.persist import push_run, run_ids

from src.config import estimation_counts, eval_batch_size, introspection_points

cfg = load(None if USE_TEST_CONFIG else SETTINGS_FILE, use_test=USE_TEST_CONFIG)
mh, rule = cfg["model_hyperparameters"], cfg["decision_rule"]
points = introspection_points(cfg)
print(f"settings      {cfg['_source']}")
print(f"rule          {rule['type']}, {rule['option_count']} options, data {cfg['model_training']['data_dir']}")
print(f"model         {mh['model_name']}, {mh['quantization']}, {cfg['compute']['mixed_precision']} on {cfg['compute']['device']}")
print(f"decisions     {mh['training_steps']} steps, a checkpoint every {mh['checkpoint_every']}, "
      f"baseline {'on' if mh['baseline_before_training'] else 'off'}")
print(f"introspection " + (f"from decision steps {points}, {mh['introspection_steps']} steps per fold, "
                           f"targets {mh['introspection_targets']}"
                           + (", decisions re-measured after" if mh["introspection_remeasure_decisions"] else "")
                           if points else "off"))
print(f"learning rate {mh['learning_rate']}, schedule {mh['lr_schedule'] or 'constant'}")
questions = cfg["report_schema"].get("questions")
if questions:
    print("questions     " + ", ".join(f"{q} ({'asked, trained' if f.get('train', True) else 'asked only'})"
                                          for q, f in questions.items() if f.get("ask", True)))
print(f"estimation    {estimation_counts(cfg)} trials per persona")
print(f"eval batches  decision {eval_batch_size(cfg, 'decision')}, introspection {eval_batch_size(cfg, 'introspection')}")

RESUME = {"run_id": RESUME_RUN_ID, "from_step": RESUME_FROM_STEP} if RESUME_RUN_ID else None
print(f"resume        " + (f"{RESUME_RUN_ID} from decision step {RESUME_FROM_STEP}" if RESUME else "off: a new run"))

HAD = set(run_ids())
try:
    rows = run(cfg, resume=RESUME)
except Exception:
    # The run stopped part way. Results are appended as a run goes, so what it wrote is worth keeping, and a
    # rented machine may be gone before anyone looks: save it now, then show the error that stopped the run.
    for rid in ([RESUME_RUN_ID] if RESUME else sorted(set(run_ids()) - HAD)):
        print(f"\nthe run stopped part way: saving what {rid} wrote so far")
        try:
            push_run(rid, HUB_REPO)
        except Exception as err:
            print(f"  {rid} NOT saved ({type(err).__name__}: {err}): run the save cells below once that is fixed")
    raise

## Save it off the machine

Results are small and worth pushing often. Adapters are tens of megabytes, so pass
`checkpoints=False` for a quick repeat push mid-run.

In [ ]:
from src.persist import push_run

push_run(rows[-1]["run_id"], HUB_REPO)

In [ ]:
# Before you walk away: sweep everything still on disk, including any run that was
# interrupted part way. Results are appended as a run goes, so a partial run is
# still worth keeping.
from src.persist import run_ids, push_run

for rid in run_ids():
    push_run(rid, HUB_REPO)